# Simple fibril model parameter sweeps

## Notebook setup

### Imports

In [3]:
# Imports
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import open3d as o3d
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm
import sys
import json
from itertools import product


from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict

sys.path.append('/Users/andrew/Library/CloudStorage/OneDrive-UCB-O365/research/data_analysis/rsoxs_suite/morph_gen/custom_fibril_gen')
from custom_fibril_gen_v2 import *

### Define Paths

In [2]:
basePath = pathlib.Path('/Users/andrew/Library/CloudStorage/OneDrive-UCB-O365/research/data_analysis/rsoxs_suite/morph_gen')
savePath = basePath.joinpath('simple_fibril_open3d_outputs')
fibrilsPath = basePath.joinpath('custom_fibril_gen', f'fixed_fibrils')

## Functions

In [98]:
# Define simple functions
def to_s0to1(value, min=0, max=360):
    """
    Normalize a scalar or array of values from a linear range [min, max]
    to the unit interval [0, 1], enforcing angle wrapping for robustness.
    """
    value = np.asarray(value)
    wrapped = (value - min) % (max - min)
    return wrapped / (max - min)

def from_s0to1(value, min=0, max=360):
    """
    Inverse of `to_s0to1`: map value(s) in [0, 1] to the linear scale [min, max]
    """
    return value * (max - min) + min

def mol_orient(scenario, rotated_bb_coords, rotated_bb_axs, rotated_bb_css, rotated_fibril, set_theta):
    """
    Assigns ZY Euler orientation angles (psi, theta) for molecular extraordinary axes
    in different alignment scenarios relative to the local fibril axis.

    Parameters:
    - scenario: str
        Molecular extraordinary axis orientation scenario identifier:
          A = Type II (para. to long axis): theta=axis_theta, psi=axis_psi
          B = Type I  (perp. to long axis): Set declination; theta=set_theta,  psi=axis_psi+90°
          C = Type I  (perp. to long axis): Radial; always point away from center of fibril cross sections         
          D = Type I  (perp. to long axis): Random; always point in the plane of the fibril cross sections
    - rotated_bb_coords: (N, 3) np.ndarray
        Backbone centerline positions along fibril long axis after rotation
    - rotated_bb_axs: (N, 3) np.ndarray
        Backbone axial unit vectors (long axis) after rotation
    - rotated_bb_css: (N, 2, 3) np.ndarray
        Backbone short-axis vectors (cross-sectional frame, 2 vectors per backbone point)
    - rotated_fibril: (M, 3) np.ndarray
        3D fibril coordinate positions (Å)
    - set_theta: int or float
        Used in Scenario B to specify declination angle

    Returns:
    - fibril_ZYrots: (M, 2) np.ndarray
        Euler ZY rotation angles (psi ∈ [0,360), theta ∈ [0,180]) per fibril point in degrees
    """
    M = rotated_fibril.shape[0]
    fibril_ZYrots = np.zeros((M, 2))
    rng = np.random.default_rng()

    if scenario == 'A':
        psi = np.rad2deg(np.arctan2(rotated_bb_axs[:, 1], rotated_bb_axs[:, 0])) % 360
        theta = np.rad2deg(np.arccos(rotated_bb_axs[:, 2])) % 180
        euler_ZY_rot = np.vstack((psi, theta)).T
        for i, fibril_point in enumerate(rotated_fibril):
            d = np.linalg.norm(rotated_bb_coords - fibril_point, axis=1)
            idx = np.argmin(d)
            fibril_ZYrots[i] = euler_ZY_rot[idx]
            
    elif scenario == 'B':
        for i, fibril_point in enumerate(rotated_fibril):
            d = np.linalg.norm(rotated_bb_coords - fibril_point, axis=1)
            idx = np.argmin(d)
            axis_vec = rotated_bb_axs[idx]
            psi = (np.rad2deg(np.arctan2(axis_vec[1], axis_vec[0])) + 90) % 360
            fibril_ZYrots[i] = [psi, set_theta]
    
    elif scenario == 'C': 
        for i, fibril_point in enumerate(rotated_fibril):
            bb_displacements = rotated_bb_coords - fibril_point
            bb_distances = np.linalg.norm(bb_displacements, axis=1)
            bb_min_idx = np.argmin(bb_distances)

            min_displacement = bb_displacements[bb_min_idx]
            orient_vec = -min_displacement / np.linalg.norm(min_displacement)

            theta = np.rad2deg(np.arccos(orient_vec[2])) % 180
            psi = np.rad2deg(np.arctan2(orient_vec[1], orient_vec[0])) % 360
            fibril_ZYrots[i] = [psi, theta]

    elif scenario == 'D': 
        for i, fibril_point in enumerate(rotated_fibril):
            dists = np.linalg.norm(rotated_bb_coords - fibril_point, axis=1)
            idx = np.argmin(dists)

            css1 = rotated_bb_css[0, idx]
            css2 = rotated_bb_css[1, idx]

            alpha = rng.uniform(0, 2 * np.pi)
            orient_vec = np.cos(alpha) * css1 + np.sin(alpha) * css2
            orient_vec = orient_vec / np.linalg.norm(orient_vec)

            theta = np.rad2deg(np.arccos(orient_vec[2])) % 180
            psi = np.rad2deg(np.arctan2(orient_vec[1], orient_vec[0])) % 360
            fibril_ZYrots[i] = [psi, theta]
    
    else:
        raise ValueError(f"Unknown scenario '{scenario}'")

    return fibril_ZYrots


## Define default model arguments

In [77]:
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from collections import defaultdict

def generate_pointcloud(n=3000):
    points = np.random.rand(n, 3)
    colors = np.random.rand(n, 3)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors)
    return pcd

def generate_voxelgrid(pcd, voxel_size=0.1):
    return o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=voxel_size)

def style_axes(ax, background, hide_all=True):
    if hide_all:
        ax.set_axis_off()
    else:
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.grid(False)
        ax.xaxis.pane.set_visible(False)
        ax.yaxis.pane.set_visible(False)
        ax.zaxis.pane.set_visible(False)

def draw_quiver_axes(ax, origin=(0, 0, 0), length=0.1):
    origin = np.atleast_2d(origin)
    U = np.array([[length, 0, 0], [0, length, 0], [0, 0, length]])
    X = np.repeat(origin, 3, axis=0)
    ax.quiver(X[:, 0], X[:, 1], X[:, 2], U[:, 0], U[:, 1], U[:, 2], 
              color=['g', 'r', 'b'], arrow_length_ratio=0.3)
    ax.text(origin[0, 0] + length * 1.1, origin[0, 1], origin[0, 2], 'X', color='g')
    ax.text(origin[0, 0], origin[0, 1] + length * 1.1, origin[0, 2], 'Y', color='r')
    ax.text(origin[0, 0], origin[0, 1], origin[0, 2] + length * 1.1, 'Z', color='b')

def set_equal_aspect_3d(ax, X, Y, Z, zoom=1.0):
    max_range = np.array([X.max() - X.min(), Y.max() - Y.min(), Z.max() - Z.min()]).max() / 2.0
    max_range *= zoom
    mid_x = (X.max() + X.min()) * 0.5
    mid_y = (Y.max() + Y.min()) * 0.5
    mid_z = (Z.max() + Z.min()) * 0.5
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    ax.set_box_aspect([1, 1, 1])

def plot_pointcloud(pcd, background="white", elev=45, azim=315, show=True, savePath=None, zoom=1.0, axes_origin=(0, 0, 0), axes_length=1, hide_axes=True):
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors) if pcd.has_colors() else "k"

    fig = plt.figure(figsize=(6, 6))
    fig.set(tight_layout=True)
    ax = fig.add_subplot(111, projection='3d')
    ax.scatter(points[:, 0], points[:, 1], points[:, 2], s=2, c=colors)

    set_equal_aspect_3d(ax, points[:, 0], points[:, 1], points[:, 2], zoom)
    draw_quiver_axes(ax, origin=axes_origin, length=axes_length)

    ax.set_facecolor(background)
    fig.patch.set_facecolor(background)
    ax.view_init(elev=elev, azim=azim)

    style_axes(ax, background, hide_all=hide_axes)

    if show:
        plt.show()
    if savePath:
        fig.savefig(savePath, dpi=250)
    # plt.close()

def plot_voxelgrid(voxel_grid, background="white", elev=45, azim=315, show=True, savePath=None, zoom=1.0, axes_origin=(0, 0, 0), axes_length=1, hide_axes=True):
    voxel_size = voxel_grid.voxel_size
    origin = voxel_grid.origin
    voxels = voxel_grid.get_voxels()

    indices = np.stack([v.grid_index for v in voxels])
    colors = np.stack([v.color for v in voxels])
    centers = indices * voxel_size + origin

    fig = plt.figure(figsize=(6, 6))
    fig.set(tight_layout=True)
    ax = fig.add_subplot(111, projection='3d')
    for center, color in zip(centers, colors):
        ax.bar3d(center[0], center[1], center[2],
                 voxel_size, voxel_size, voxel_size,
                 color=color, edgecolor='k', linewidth=0.1, alpha=0.9)

    set_equal_aspect_3d(ax, centers[:, 0], centers[:, 1], centers[:, 2], zoom)
    draw_quiver_axes(ax, origin=axes_origin, length=axes_length)

    ax.set_facecolor(background)
    fig.patch.set_facecolor(background)
    ax.view_init(elev=elev, azim=azim)

    style_axes(ax, background, hide_all=hide_axes)

    if show:
        plt.show()
    if savePath:
        fig.savefig(savePath, dpi=250)
    # plt.close()

# # Run example plots
# pcd = generate_pointcloud()
# voxel_grid = generate_voxelgrid(pcd)

# plot_pointcloud(pcd)
# plot_voxelgrid(voxel_grid)

In [100]:
np.arange(0,100,10).tolist()

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

In [1]:
args = {}

# mol_orient_list = ['A', 'B', 'C', 'D', 'E']


args['params_ID'] = 'testing'  # put a unique identifier for the default arguments entered below
args['sweep_params'] =        ['molecular_orientation',           'fibril_tilt']
args['sweep_params_values'] = [        ['A', 'C', 'D'], [0, 15, 25, 75, 85, 90]]  # array of values to sweep specified parameter over

# Arguments for an individual fibril, all length units in Angstroms
args['fibril_density'] = 10 / (10**3)  # particles per volume
args['fibril_length'] = 600
args['fibril_diam'] = 40
args['fibril_flex'] = 1e-10  # flexibility value given as gaussian std deviation of angle about 0 in radians when creating fibril backbone: 1e-4 = cylinder, 1e-1 = spaghetti
args['fibril_fuzz_dens'] = 1 / (10**3)  # particles per volume
args['fibril_fuzz_length'] =  0 # length (radius) of fuzziness. ex: 10Å side chains extending from polymer

# Arguments for molecular/fibril orientation:
args['molecular_orientation'] = 'B'  # Input orienation scenario str
args['set_theta'] = 45  # Optional parameter that is only used in orientation scenario B for the fixed declination angle
args['fibril_tilt'] = 5  # Tilt in degrees away from z axis (substrate normal axis)
args['S'] = 1  # Orientation strength at center of fibril
args['S_slope_per_nm'] = 0  # Introduce a linear decay of S moving away from center of fibril

# Arguments for voxel grid setup
args["PhysSize_angstroms_per_voxel"] = 10
args["ld"] = 256
args["vd"] = 128
args['fibril_center_vd'] = 64  # Central Z voxel position to move fibril to
args['fibril_center_ld'] = 128  # Central XY voxel position to move fibril to

display(args)

{'params_ID': 'testing',
 'sweep_params': ['molecular_orientation', 'fibril_tilt'],
 'sweep_params_values': [['A', 'C', 'D'], [0, 15, 25, 75, 85, 90]],
 'fibril_density': 0.01,
 'fibril_length': 600,
 'fibril_diam': 40,
 'fibril_flex': 1e-10,
 'fibril_fuzz_dens': 0.001,
 'fibril_fuzz_length': 0,
 'molecular_orientation': 'B',
 'set_theta': 45,
 'fibril_tilt': 5,
 'S': 1,
 'S_slope_per_nm': 0,
 'PhysSize_angstroms_per_voxel': 10,
 'ld': 256,
 'vd': 128,
 'fibril_center_vd': 64,
 'fibril_center_ld': 128}

## Build and save fibril PointClouds

In [4]:
sweep_keys = args['sweep_params']
sweep_vals = args['sweep_params_values']
combinations = list(product(*sweep_vals))

# params_ID = sweep_args['params_ID']
# sweepPath = savePath.joinpath(f'paramsID-{params_ID}_sweep-{sweep_keys[0]}-{sweep_keys[1]}')
# sweepPath.mkdir(exist_ok=True)

for combo in tqdm(combinations, desc="Sweeping params"):
    sweep_args = args.copy()
    for k, v in zip(sweep_keys, combo):
        sweep_args[k] = v

    # Create fibril
    backbone_coords, backbone_axs, backbone_css, fibril_coords = gen_scat_coords_flexcyl(
        sweep_args['fibril_density'],
        sweep_args['fibril_length'],
        sweep_args['fibril_diam'],
        sweep_args['fibril_flex'],
        sweep_args['fibril_fuzz_dens'],
        sweep_args['fibril_fuzz_length']
    )

    # Rotate fibril
    rotation = R.from_euler('yz', [sweep_args['fibril_tilt'], 0], degrees=True)
    rotated_bb = rotation.apply(backbone_coords)
    rotated_fibril = rotation.apply(fibril_coords)
    rotated_bb_axs = rotation.apply(backbone_axs)
    rotated_bb_css_1 = rotation.apply(backbone_css[:, 0, :])
    rotated_bb_css_2 = rotation.apply(backbone_css[:, 1, :])
    rotated_bb_css = np.stack((rotated_bb_css_1, rotated_bb_css_2))

    # Set molecular orientations
    fibril_ZYrots = mol_orient(sweep_args['molecular_orientation'], rotated_bb, rotated_bb_axs, rotated_bb_css, rotated_fibril, set_theta = sweep_args['set_theta'])
    fibril_RGB_values = np.array([
        np.sin(np.deg2rad(fibril_ZYrots[:,0]))**2,  # psi, inverse: np.rad2deg(np.arcsin(np.sqrt(fibril_RGB_values[:,0])))
        np.sin(np.deg2rad(fibril_ZYrots[:,1]))**2,  # theta, inverse: np.rad2deg(np.arcsin(np.sqrt(fibril_RGB_values[:,1])))
        # to_s0to1(fibril_ZYrots[:, 0], min=-180, max=180),  # psi
        # to_s0to1(fibril_ZYrots[:, 1], min=0, max=180),  # theta
        np.full((fibril_ZYrots[:, 0].shape), sweep_args['S'])
    ]).T


    # Make PointCloud
    pcd = o3d.geometry.PointCloud()
    pcd.points.extend(rotated_fibril)
    pcd.colors.extend(fibril_RGB_values)

    # Voxelize PointCloud
    vg = o3d.geometry.VoxelGrid.create_from_point_cloud(pcd, voxel_size=sweep_args["PhysSize_angstroms_per_voxel"])

    # Save VoxelGrid and Visualizations
    label = "_".join([f"{k}-{v}" for k, v in zip(sweep_keys, combo)])

    # o3d.io.write_voxel_grid(str(sweepPath.joinpath(f'{label}.ply')), vg)
    # plot_pointcloud(pcd, show=True, axes_origin=(-100, -100, 0), axes_length=90,  savePath=sweepPath.joinpath(f'{label}_pcd.png'), zoom=1, hide_axes=True)
    # plot_voxelgrid(vg, show=True, axes_origin=(-100, -100, 0), axes_length=90,  savePath=sweepPath.joinpath(f'{label}_vg.png'), zoom=1, hide_axes=True)
    plot_pointcloud(pcd, show=True, axes_origin=(-100, -100, 0), axes_length=90, zoom=1, hide_axes=True)
    plot_voxelgrid(vg, show=True, axes_origin=(-100, -100, 0), axes_length=90, zoom=1, hide_axes=True)
    plt.close('all')

# # Save arguments dictionary
# json_ready_args = {
#     k: v.tolist() if isinstance(v, np.ndarray) else v
#     for k, v in args.items()
# }
# with sweepPath.joinpath('args.json').open('w') as f:
#     json.dump(json_ready_args, f)


Sweeping params:   0%|                                                            | 0/18 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 4)

In [74]:
fibril_ZYrots

array([[89.99999991, 90.        ],
       [89.99999987, 90.        ],
       [89.99999991, 90.        ],
       ...,
       [89.99999987, 90.        ],
       [89.99999993, 90.        ],
       [89.99999997, 90.        ]])

## Arrange fibrils

In [164]:
fibril_ZYrots.shape

(7527, 2)

In [172]:
fibril_ZYrots[:,1].max()

90.15628476268236

In [166]:
o3d.visualization.EV.set(pc)

True

## Use open3D to voxelize point cloud

In [ ]:
voxel_grid = o3d.geometry.VoxelGrid.create_from_point_cloud(pc, voxel_size=10)
voxel_grid

In [ ]:
# Convert voxel grid to list of voxels, with grid index & color

voxels = voxel_grid.get_voxels()  # returns list of voxels
indices = np.stack(list(vx.grid_index for vx in voxels))
colors = np.stack(list(vx.color for vx in voxels))

voxels

In [ ]:
indices[:,0].max()

In [ ]:
# visualize, will need to restart kernel after :O
o3d.visualization.draw_geometries([voxel_grid])

## Save open3d voxelgrids